In [0]:
import time

dbutils.widgets.text("catalog", "dev", "Catalog Name")
CATALOG = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog: {CATALOG}")
print(f"Tables we'll optimize:")
print(f"  - {CATALOG}.gold.fact_sales")
print(f"  - {CATALOG}.silver.silver_orders")
print(f"  - {CATALOG}.silver.silver_customers")

OPTIMIZE with Z-ORDER

In [0]:
%sql
OPTIMIZE dev.gold.fact_sales
ZORDER BY (customer_id, product_id);

In [0]:
%sql
-- Optimize silver orders by customer_id (common join key)
OPTIMIZE dev.silver.silver_orders
ZORDER BY (customer_id);

In [0]:
%sql
-- Optimize silver order_items by order_id and product_id
OPTIMIZE dev.silver.silver_order_items
ZORDER BY (order_id, product_id);

In [0]:
%sql
-- Optimize enriched table
OPTIMIZE dev.silver.silver_orders_enriched
ZORDER BY (customer_id, product_id);

### Run VACUUM

In [0]:
%sql
VACUUM dev.gold.fact_sales RETAIN 168 HOURS;
VACUUM dev.silver.silver_orders RETAIN 168 HOURS;
VACUUM dev.silver.silver_order_items RETAIN 168 HOURS;
VACUUM dev.silver.silver_customers RETAIN 168 HOURS;

In [0]:
start = time.time()
result = spark.sql(f"""
    SELECT customer_id, COUNT(*) as orders, SUM(line_total) as revenue
    FROM {CATALOG}.gold.fact_sales
    WHERE customer_id = 1001
    GROUP BY customer_id
""").collect()
elapsed = time.time() - start

print(f"Query on Z-ORDERed column (customer_id = 1001): {elapsed:.2f}s")
print(f"Z-ORDER means Spark skips files that don't contain customer 1001")
print(f"Without Z-ORDER, Spark would scan ALL files")

In [0]:
from pyspark.sql.functions import date_format

# Create a partitioned version of fact_sales
df_fact = spark.table(f"{CATALOG}.gold.fact_sales")

df_partitioned = df_fact.withColumn("order_month", date_format("order_date", "yyyy-MM"))

target = f"{CATALOG}.gold.fact_sales_partitioned"
(df_partitioned.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("order_month")
 .option("overwriteSchema", "true")
 .saveAsTable(target))

partition_count = df_partitioned.select("order_month").distinct().count()
print(f"Created partitioned table: {target}")
print(f"Total partitions (months): {partition_count}")
import time

# Query 1: Non-partitioned — scans ALL files
start = time.time()
r1 = spark.sql(f"""
    SELECT COUNT(*) as cnt, SUM(line_total) as revenue
    FROM {CATALOG}.gold.fact_sales
    WHERE order_date BETWEEN '2024-06-01' AND '2024-06-30'
""").collect()
t1 = time.time() - start
print(f"Non-partitioned query: {t1:.2f}s | Revenue: {r1[0]['revenue']}")

# Query 2: Partitioned — reads ONLY June 2024 partition
start = time.time()
r2 = spark.sql(f"""
    SELECT COUNT(*) as cnt, SUM(line_total) as revenue
    FROM {CATALOG}.gold.fact_sales_partitioned
    WHERE order_month = '2024-06'
""").collect()
t2 = time.time() - start
print(f"Partitioned query:     {t2:.2f}s | Revenue: {r2[0]['revenue']}")

print(f"\nPartition pruning speedup: {t1/t2:.2f}x faster")
print(f"Spark skipped {partition_count - 1} out of {partition_count} partitions!")
